# 📊 Algorix — Video Exploratory Data Analysis (EDA)
**WIUT Hackathon 2026 — Computer Vision**

Ushbu notebook sample videolarning quyidagi xususiyatlarini tahlil qiladi:
1. **Video metrikalari:** Resolution, FPS, Duration, Frame count
2. **Yorug'lik sharoiti:** Brightness (Luma), Contrast, Day/Night/Dusk klassifikatsiyasi
3. **Scene Geometry:** `camera.md` bo'yicha yo'laklar, stop-line, piyodalar yo'lagi
4. **Motion Heatmap & Density:** Harakat intensivligi va tirbandlik nuqtalari
5. **Ground Truth (GT):** Dev-set hodisalari taqsimoti

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()))

from src.utils.determinism import set_global_seed
from src.geometry.camera_parser import load_scene_geometry
from labeling.label_tool import DatasetLabelManager
from eda.video_analyzer import VideoAnalyzer

set_global_seed(42)
print("✅ Setup muvaffaqiyatli yakunlandi.")

## 1. Scene Geometriyasini yuklash va tekshirish

In [ ]:
scene = load_scene_geometry()
print(f"Resolution: {scene.resolution}")
print(f"Nominal FPS: {scene.fps}")
print(f"Yo'laklar soni: {len(scene.lanes)}")
print(f"Stop-linelar soni: {len(scene.stop_lines)}")
print(f"Piyodalar o'tish hududlari: {len(scene.crossings)}")

## 2. Samples papkasidagi videolarni tahlil qilish

In [ ]:
samples_dir = Path("samples")
video_files = list(samples_dir.glob("*.mp4")) + list(samples_dir.glob("*.avi"))

analyzer = VideoAnalyzer(sample_interval_frames=30)
results = {}

if not video_files:
    print("⚠️ Hozircha 'samples/' papkasida video mavjud emas. Video yuklangach avtomatik tahlil qilinadi.")
else:
    for vf in video_files:
        meta = analyzer.analyze_video(str(vf))
        results[vf.name] = meta
        print(f"Video: {vf.name} -> {meta['width']}x{meta['height']} @ {meta['fps']} FPS, {meta['duration_sec']}s, {meta.get('lighting', {}).get('condition', 'N/A')}")

## 3. Ground Truth (GT) Dev-set annotatsiyalari

In [ ]:
manager = DatasetLabelManager()
gt_files = list(manager.labels_dir.glob("*_gt.json"))
print(f"Mavjud GT fayllar: {len(gt_files)} ta")
for gtf in gt_files:
    events = manager.load_gt(gtf.stem.replace("_gt", ""))
    print(f"- {gtf.name}: {len(events)} ta event")